In [34]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer


In [2]:
data = pd.read_csv('data.csv')

In [23]:
data.columns

Index(['projectid', 'school_state', 'school_metro', 'school_magnet',
       'school_nlns', 'school_kipp', 'school_charter',
       'school_charter_ready_promise', 'teacher_teach_for_america',
       'teacher_ny_teaching_fellow', 'primary_focus_subject',
       'primary_focus_area', 'resource_type', 'poverty_level', 'grade_level',
       'total_price_excluding_optional_support',
       'total_price_including_optional_support', 'students_reached',
       'eligible_double_your_impact_match', 'eligible_almost_home_match',
       'date_posted', 'fully_funded', 'short_description', 'need_statement',
       'essay', 'students_reached_scaled',
       'total_price_excluding_optional_support_scaled', 'school_state_region'],
      dtype='object')

In [24]:
data['fully_funded'].value_counts()

fully_funded
1    346981
0    138067
Name: count, dtype: int64

**School Metro & Poverty Level**

Baseline

In [4]:
features = ['school_metro','poverty_level']

In [ ]:
ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

In [8]:
nb = GaussianNB()

In [9]:
nb.fit(X_train,y_train)

GaussianNB()

In [10]:
y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.37      0.28      0.32     27614
           1       0.74      0.81      0.77     69396

    accuracy                           0.66     97010
   macro avg       0.55      0.54      0.54     97010
weighted avg       0.63      0.66      0.64     97010

Specificity: 0.28
ROC-AUC: 0.5438533167206625



Handling unbalance classes with Random Under Sampler

In [15]:
from imblearn.under_sampling import RandomUnderSampler

In [16]:
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


In [20]:
nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.35      0.42      0.38     27614
           1       0.75      0.69      0.72     69396

    accuracy                           0.61     97010
   macro avg       0.55      0.55      0.55     97010
weighted avg       0.63      0.61      0.62     97010

Specificity: 0.42
ROC-AUC: 0.5438533167206625



**Add Resource Type and Primary Subject**

Baseline

In [81]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [82]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.35      0.60      0.44     27614
           1       0.78      0.55      0.65     69396

    accuracy                           0.57     97010
   macro avg       0.56      0.58      0.54     97010
weighted avg       0.65      0.57      0.59     97010

Specificity: 0.60
ROC-AUC: 0.5750178610758059



With RUS

In [83]:
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.32      0.77      0.46     27614
           1       0.80      0.36      0.49     69396

    accuracy                           0.48     97010
   macro avg       0.56      0.57      0.48     97010
weighted avg       0.66      0.48      0.48     97010

Specificity: 0.77
ROC-AUC: 0.5750178610758059



In [86]:
#checking distribution

print(f"\nBefore Undersampling - Training examples: {len(X_train)}")
print(f"Class distribution: {np.bincount(y_train)}")

print(f"\nAfter Undersampling - Training examples: {len(X_train_rus)}")
print(f"Class distribution: {np.bincount(y_train_rus)}")


Before Undersampling - Training examples: 388038
Class distribution: [110453 277585]

After Undersampling - Training examples: 220906
Class distribution: [110453 110453]


Add Students Reached + Funding Request Amt

Baseline

In [80]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [ ]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data[features])

features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data.loc[i,'students_reached']] + [data.loc[i,'total_price_excluding_optional_support']]


features_encoded = np.array(features_encoded)

array([[  0.  ,   0.  ,   1.  , ...,   0.  ,  20.  , 574.  ],
       [  0.  ,   0.  ,   1.  , ...,   0.  ,  45.  , 258.3 ],
       [  0.  ,   0.  ,   1.  , ...,   0.  ,  25.  , 236.88],
       ...,
       [  0.  ,   1.  ,   0.  , ...,   0.  ,  29.  , 537.5 ],
       [  1.  ,   0.  ,   0.  , ...,   0.  , 120.  , 187.81],
       [  0.  ,   1.  ,   0.  , ...,   0.  ,   7.  , 444.36]])

In [78]:
#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data['fully_funded'], test_size=0.2, random_state=42,stratify=data['fully_funded'])

nb = GaussianNB()
nb.fit(X_train,y_train)

#testing

y_pred = nb.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.37      0.61      0.46     27614
           1       0.79      0.59      0.68     69396

    accuracy                           0.60     97010
   macro avg       0.58      0.60      0.57     97010
weighted avg       0.67      0.60      0.62     97010

Specificity: 0.61
ROC-AUC: 0.5995966571316727



With RUS

In [79]:
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

nb_rus = GaussianNB()
nb_rus.fit(X_train_rus,y_train_rus)
y_pred_rus = nb_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.34      0.77      0.47     27614
           1       0.82      0.41      0.55     69396

    accuracy                           0.51     97010
   macro avg       0.58      0.59      0.51     97010
weighted avg       0.68      0.51      0.53     97010

Specificity: 0.77
ROC-AUC: 0.5995966571316727

